# ADS HW2: Regression & Classification Modeling

Building on HW1 (EDA + feature engineering) to train, tune, and compare regression and classification models on the Telco Customer Churn dataset. The notebook emphasizes metric choice, model comparison, and clarity of explanations.

- Regression target: `total_charges` (continuous)
- Binary classification target: `churn flag` (Yes/No)
- Multiclass target: `PaymentMethod` (4 classes)
- Dataset source: `data/telco_customer_churn.csv` (or Kaggle equivalent)



In [ ]:
!pip install catboost opendatasets

In [ ]:
import warnings
from pathlib import Path
import subprocess
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    mean_absolute_percentage_error,
    r2_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    log_loss,
    precision_recall_curve,
)

# Models
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.svm import SVR, SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
import xgboost as xgb
import lightgbm as lgb
import catboost as cb

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
plt.rcParams["figure.figsize"] = (12, 6)
sns.set_theme(style="whitegrid")


In [ ]:
# import os
# from pathlib import Path

# # Kaggle dataset info
# dataset_url = "https://www.kaggle.com/datasets/beatafaron/telco-customer-churn-realistic-customer-feedback"
# kaggle_input_dir = Path("/kaggle/input/telco-customer-churn-realistic-customer-feedback")
# download_dir = Path("telco-customer-churn-realistic-customer-feedback")

# # 1) Prefer using the mounted Kaggle input directory
# if kaggle_input_dir.is_dir():
#     data_dir = kaggle_input_dir
#     print(f"Using dataset from Kaggle input: {data_dir}")
# else:
#     # 2) Fallback: download (for Colab/local use, etc.)
#     try:
#         import opendatasets as od

#         if not download_dir.exists():
#             print("Downloading dataset from Kaggle...")
#             od.download(dataset_url)

#         data_dir = download_dir
#         print(f"Using downloaded dataset at: {data_dir}")
#     except ImportError:
#         raise RuntimeError(
#             "Dataset not found in /kaggle/input and opendatasets is not installed. "
#             "Attach the Kaggle dataset or install opendatasets to download it."
#         )

# # Optional: list CSV files in the chosen directory
# kaggle_csvs = list(data_dir.glob("*.csv"))
# print(f"Found Kaggle CSVs: {kaggle_csvs}")


In [ ]:
# import os
# import pandas as pd
# from pathlib import Path

# # Kaggle dataset directory
# KAGGLE_DIR = Path("/kaggle/input/telco-customer-churn-realistic-customer-feedback")

# # Choose which CSV to load from the dataset
# DATA_FILENAME = "telco_prep.csv"   # <- change if needed

# # Try Kaggle first
# if KAGGLE_DIR.exists():
#     DATA_PATH = KAGGLE_DIR / DATA_FILENAME
#     print(f"Loading dataset from Kaggle input: {DATA_PATH}")
# else:
#     # Fallback for local development
#     LOCAL_DIR = Path("data")
#     DATA_PATH = LOCAL_DIR / DATA_FILENAME
#     print(f"Loading dataset from local path: {DATA_PATH}")

# df_raw = pd.read_csv(DATA_PATH)

# # If you have a feature engineering function defined earlier, apply it:
# try:
#     df = feature_engineer_data(df_raw)
# except NameError:
#     df = df_raw  # no feature engineering function defined

# print("Data Shape:", df.shape)
# df.head()


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("beatafaron/telco-customer-churn-realistic-customer-feedback")

print("Path to dataset files:", path)

# Do not delete below code (even thought it is duplicate it is important)

In [ ]:
# import kagglehub

# # Download latest version
# path = kagglehub.dataset_download("beatafaron/telco-customer-churn-realistic-customer-feedback")

# print("Path to dataset files:", path)

In [ ]:
import pandas as pd
from pathlib import Path

# KAGGLE_DIR = Path("/kaggle/input/telco-customer-churn-realistic-customer-feedback")
KAGGLE_DIR = Path("/root/.cache/kagglehub/datasets/beatafaron/telco-customer-churn-realistic-customer-feedback/versions/8")
DATA_FILENAME = "telco_churn_with_all_feedback.csv"   # <-- choose the file you want

# 1) If running on Kaggle, use the mounted dataset
if KAGGLE_DIR.exists():
    DATA_PATH = KAGGLE_DIR / DATA_FILENAME
    print(f"Loading dataset from Kaggle input: {DATA_PATH}")

# 2) Otherwise fall back to local folder for non-Kaggle execution
else:
    DATA_PATH = Path("data") / DATA_FILENAME
    print(f"Loading dataset from local directory: {DATA_PATH}")

df_raw = pd.read_csv(DATA_PATH)

# If feature_engineer_data() exists, use it; else just continue
try:
    df = feature_engineer_data(df_raw)
except NameError:
    df = df_raw

print("Data shape:", df.shape)
df.head()

## 2) Preprocessing setup
- Define numeric/categorical feature lists for reuse across tasks.
- Use `SimpleImputer` + `StandardScaler` for numeric columns.
- Use `SimpleImputer` + `OneHotEncoder` for categorical columns (handle unknowns).
- Build helper evaluators for regression, binary, and multiclass tasks.



In [ ]:
# Feature groups
numeric_cols = [
    "tenure",
    "monthly_charges",
    "services_count",
    "contract_months",
    "avg_revenue_per_month",
    "tenure_years",
    "tenure_quarter",
    "services_ratio",
    "avg_charge_per_service",
    "revenue_vs_contract",
]

categorical_cols = [
    "gender",
    "partner",
    "dependents",
    "phone_service",
    "multiple_lines",
    "internet_service",
    "online_security",
    "online_backup",
    "device_protection",
    "tech_support",
    "streaming_tv",
    "streaming_movies",
    "contract",
    "paperless_billing",
    "PaymentMethod",
    "tenure_bucket",
    "monthly_charge_band",
]

# Keep only columns that exist
numeric_cols = [c for c in numeric_cols if c in df.columns]
categorical_cols = [c for c in categorical_cols if c in df.columns]

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "encoder",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
        ),
    ]
)

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

def make_preprocessor():
    return ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_cols),
            ("cat", categorical_transformer, categorical_cols),
        ]
    )



In [ ]:
def evaluate_regression(model, X_train, y_train, X_test, y_test, name: str):
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    return {
        "model": name,
        "mse": mean_squared_error(y_test, preds),
        "mae": mean_absolute_error(y_test, preds),
        "mape": mean_absolute_percentage_error(y_test, preds),
        "r2": r2_score(y_test, preds),
    }


def evaluate_binary(model, X_train, y_train, X_test, y_test, name: str):
    model.fit(X_train, y_train)
    prob = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else model.decision_function(X_test)
    preds = (prob >= 0.5).astype(int)

    precision, recall, _ = precision_recall_curve(y_test, prob)
    pr_auc = np.trapz(recall, precision)

    return {
        "model": name,
        "accuracy": accuracy_score(y_test, preds),
        "precision": precision_score(y_test, preds),
        "recall": recall_score(y_test, preds),
        "f1": f1_score(y_test, preds),
        "roc_auc": roc_auc_score(y_test, prob),
        "pr_auc": pr_auc,
        "confusion": confusion_matrix(y_test, preds),
    }


def evaluate_multiclass(model, X_train, y_train, X_test, y_test, name: str):
    model.fit(X_train, y_train)
    probs = model.predict_proba(X_test)
    preds = probs.argmax(axis=1)
    return {
        "model": name,
        "accuracy": accuracy_score(y_test, preds),
        "f1_macro": f1_score(y_test, preds, average="macro"),
        "f1_micro": f1_score(y_test, preds, average="micro"),
        "f1_weighted": f1_score(y_test, preds, average="weighted"),
        "log_loss": log_loss(y_test, probs),
    }



## 3) Regression: predict `total_charges`
Models: Linear Regression, Ridge, Lasso, and kernel SVR (RBF) to showcase the kernel trick. Metrics: MSE, MAE, MAPE, R². We split once and reuse the same preprocessing.



In [ ]:
target_reg = "TotalCharges"
# Fix for ValueError: could not convert string to float: ' '
df[target_reg] = pd.to_numeric(df[target_reg], errors='coerce').fillna(0)
X_reg = df.drop(columns=[c for c in ["customer_id", "churn", "TotalCharges", "log_total_charges"] if c in df.columns])
y_reg = df[target_reg]

preprocess_reg = make_preprocessor()

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

models_reg = [
    ("Linear Regression", LinearRegression()),
    ("Ridge (L2)", Ridge(alpha=1.0)),
    ("Lasso (L1)", Lasso(alpha=0.001)),
    ("Kernel SVR (RBF)", SVR(kernel="rbf", C=10, gamma="scale")),
]

reg_results = []
for name, model in models_reg:
    pipe = Pipeline(steps=[("prep", preprocess_reg), ("model", model)])
    reg_results.append(
        evaluate_regression(pipe, X_train_reg, y_train_reg, X_test_reg, y_test_reg, name)
    )

reg_df = pd.DataFrame(reg_results).set_index("model").sort_values("mse")
reg_df


**Kernel trick (brief):** Instead of explicitly creating polynomial or radial features, kernel methods (e.g., RBF SVR/SVM) compute similarity in an implicit high-dimensional space. This allows linear algorithms to fit non-linear relationships while keeping computation in the original space via kernel evaluations.



## 4) Binary classification: predict `churn flag`
Models: Logistic Regression, Linear SVM, Kernel SVM (RBF), KNN (tuned K), Decision Tree (tuned depth), Random Forest. Metrics: Accuracy, Precision, Recall, F1, ROC-AUC, PR-AUC, Confusion Matrix.



In [ ]:
# Binary classification target: use Churn column directly
# Find the churn column (case-insensitive)
churn_col = next((c for c in df.columns if c.lower() == 'churn'), None)
if churn_col is None:
    raise ValueError(f'Churn column not found. Available: {df.columns.tolist()}')

# Convert Yes/No to 1/0 for classification
y_bin = df[churn_col].apply(lambda x: 1 if str(x).lower() in ['yes', 'true', '1'] else 0)

# Drop target and ID columns from features
cols_to_drop = [c for c in df.columns if c.lower() in ['customerid', 'customer_id', 'churn', 'totalcharges', 'total_charges', 'PaymentMethod']]
X_bin = df.drop(columns=cols_to_drop)

preprocess_cls = make_preprocessor()

X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(
    X_bin, y_bin, test_size=0.2, random_state=42, stratify=y_bin
)

models_bin = [
    ("Logistic Regression", LogisticRegression(max_iter=200, class_weight="balanced")),
    ("Linear SVM", SVC(kernel="linear", probability=True, class_weight="balanced", C=1.0)),
    ("RBF SVM", SVC(kernel="rbf", probability=True, class_weight="balanced", C=2.0, gamma="scale")),
    ("KNN (k=15)", KNeighborsClassifier(n_neighbors=15)),
    ("Decision Tree (max_depth=6)", DecisionTreeClassifier(max_depth=6, random_state=42, class_weight="balanced")),
    ("Random Forest", RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, class_weight="balanced")),
]

bin_results = []
conf_mats = {}
for name, model in models_bin:
    pipe = Pipeline(steps=[("prep", preprocess_cls), ("model", model)])
    result = evaluate_binary(pipe, X_train_b, y_train_b, X_test_b, y_test_b, name)
    bin_results.append({k: v for k, v in result.items() if k != "confusion"})
    conf_mats[name] = result["confusion"]

bin_df = pd.DataFrame(bin_results).set_index("model").sort_values("f1", ascending=False)
bin_df


In [ ]:
# Plot confusion matrix for best binary model (by F1)
best_bin_model = bin_df.index[0]
cm = conf_mats[best_bin_model]

fig, ax = plt.subplots()
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax)
ax.set_title(f"Confusion Matrix – {best_bin_model}")
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
plt.show()



In [ ]:
# ROC curve for the best binary model
model_lookup = {name: model for name, model in models_bin}
best_model_est = model_lookup[best_bin_model]
best_pipe = Pipeline(steps=[("prep", preprocess_cls), ("model", best_model_est)])
best_pipe.fit(X_train_b, y_train_b)

prob_best = best_pipe.predict_proba(X_test_b)[:, 1]
fpr, tpr, _ = roc_curve(y_test_b, prob_best)
auc_score = roc_auc_score(y_test_b, prob_best)

fig, ax = plt.subplots()
ax.plot(fpr, tpr, label=f"ROC AUC = {auc_score:.3f}")
ax.plot([0, 1], [0, 1], linestyle="--", color="gray")
ax.set_title(f"ROC Curve – {best_bin_model}")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.legend()
plt.show()



## 5) Multiclass classification: predict `PaymentMethod`
Models: Multiclass Logistic Regression (OVR + Multinomial), Linear SVM, RBF SVM, KNN, Decision Tree, and gradient boosting variants (XGBoost, LightGBM, CatBoost). Metrics: Accuracy, macro/micro/weighted F1, log loss.



In [ ]:
from sklearn.preprocessing import LabelEncoder

target_multi = "PaymentMethod"
le_multi = LabelEncoder()
y_multi = le_multi.fit_transform(df[target_multi])
num_classes = len(le_multi.classes_)

X_multi = df.drop(columns=[c for c in ["customer_id", "churn", "PaymentMethod", "total_charges"] if c in df.columns])

cat_multi = [c for c in categorical_cols if c != target_multi]
preprocess_multi = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        (
            "cat",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
                ]
            ),
            cat_multi,
        ),
    ]
)

X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X_multi, y_multi, test_size=0.2, random_state=42, stratify=y_multi
)

models_multi = [
    ("LogReg OVR", LogisticRegression(max_iter=300, multi_class="ovr")),
    ("LogReg Multinomial", LogisticRegression(max_iter=300, multi_class="multinomial")),
    ("Linear SVM", SVC(kernel="linear", probability=True)),
    ("RBF SVM", SVC(kernel="rbf", probability=True, C=2.0, gamma="scale")),
    ("KNN (k=15)", KNeighborsClassifier(n_neighbors=15)),
    ("Decision Tree", DecisionTreeClassifier(max_depth=8, random_state=42)),
    (
        "XGBoost",
        xgb.XGBClassifier(
            n_estimators=200,
            max_depth=6,
            learning_rate=0.1,
            subsample=0.9,
            colsample_bytree=0.9,
            objective="multi:softprob",
            num_class=num_classes,
            eval_metric="mlogloss",
            random_state=42,
        ),
    ),
    (
        "LightGBM",
        lgb.LGBMClassifier(
            objective="multiclass",
            num_class=num_classes,
            n_estimators=300,
            learning_rate=0.1,
            random_state=42,
        ),
    ),
    (
        "CatBoost",
        cb.CatBoostClassifier(
            loss_function="MultiClass",
            iterations=200,
            depth=8,
            learning_rate=0.1,
            verbose=False,
            random_seed=42,
        ),
    ),
]

multi_results = []
for name, model in models_multi:
    pipe = Pipeline(steps=[("prep", preprocess_multi), ("model", model)])
    multi_results.append(
        evaluate_multiclass(pipe, X_train_m, y_train_m, X_test_m, y_test_m, name)
    )

multi_df = pd.DataFrame(multi_results).set_index("model").sort_values("f1_macro", ascending=False)
multi_df


## 6) Metric justifications & discussion
- **Regression metric choice:** MAPE highlights relative error and is useful for billing amounts; combine with MAE to avoid instability when targets are near zero. R² tracks explained variance but can hide systematic bias if scale errors are small.
- **Binary metric choice:** With ~26% churn, Accuracy can be misleading; prioritize F1/ROC-AUC (or PR-AUC if focusing on the churn class) to balance precision and recall under class imbalance.
- **Multiclass metric choice:** Macro-F1 treats all payment classes equally; use it when minority classes matter. Micro-F1 aligns with overall accuracy; weighted-F1 guards against small-class volatility.
- **Decision tree regularization:** limit `max_depth`, require `min_samples_split`/`min_samples_leaf`, and use cost-complexity pruning to reduce overfitting.
- **Linear vs. Kernel SVM:** Linear SVM is fast/interpretable for (near) linearly separable data; Kernel SVM (e.g., RBF) maps inputs to a higher-dimensional space via the kernel trick, capturing non-linear boundaries at higher compute cost.



## 7) Next steps
- Run k-fold cross-validation to validate stability.
- Hyperparameter search for KNN (`n_neighbors`) and tree ensembles (depth, estimators).
- Add residual plots for regression and PR curves for the churn-positive class.
- Optionally cache preprocessed matrices to speed up reruns.



# Task
Clean and convert the 'TotalCharges' column in the DataFrame `df` to a numeric type, handling non-numeric values by coercing them to `NaN` and then filling these `NaN`s using the median. Apply these preprocessing steps to `TotalCharges` before it is assigned to `y_reg` in respective cell.

## Clean and Convert 'TotalCharges' to Numeric

### Subtask:
Modify to preprocess the `TotalCharges` column by converting it to a numeric type, coercing non-numeric values to `NaN`, and then filling these `NaN`s with the median before assigning it to `y_reg`.


## Summary:

### Data Analysis Key Findings
*   The `TotalCharges` column was successfully cleaned and converted to a numeric data type, making it suitable for quantitative analysis.
*   Non-numeric values within `TotalCharges` were identified and handled by coercing them to `NaN`.
*   Missing values (`NaN`s) in the `TotalCharges` column were imputed using the median of the column, ensuring completeness for further processing.

### Insights or Next Steps
*   The preprocessed `TotalCharges` data, assigned to `y_reg`, is now prepared and ready for use in regression models or other analytical tasks requiring a clean numeric target variable.
